<style>
@page {
    size: A5;
    margin: 18mm 16mm 18mm 16mm;
}

body {
    font-family: "DejaVu Sans", "Arial", sans-serif;
    font-size: 12px;
    line-height: 1.35;
}

.jp-Notebook {
    max-width: 180mm;
    margin: 0 auto;
}

.jp-Cell {
    page-break-inside: avoid;
    break-inside: avoid;
}

pre, code {
    white-space: pre-wrap;
    word-break: break-word;
    overflow-wrap: anywhere;
}

table {
    width: 100%;
    table-layout: fixed;
    word-wrap: break-word;
}

img, svg, canvas {
    max-width: 100%;
    height: auto;
}

.output_area pre {
    white-space: pre-wrap;
    word-break: break-word;
    overflow-wrap: anywhere;
}
</style>

In [37]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as st
import seaborn as sns
import math as mt
from skimage.io import imread, imsave

np.random.seed(777)

In [38]:
def pareto(x, theta):
    return (1 - x)**(1 / (1 - theta))

def pow2theta(theta):
    return 2 ** (1/(theta - 1))

n = 100
beta = 0.95
theta = 25 # <- захотели

X = pareto(st.uniform(loc=0, scale=1).rvs(size=n), theta=theta)

**d)** Для сгенерированной выборки размера $ n = 100 $ и $ \theta = 25 $ вычислим доверительные интервалы

In [39]:
# Асимптотический доверительный интервал для медианы

theta_omp = 1 + 1 / (np.mean(np.log(X)))

med     = pow2theta(theta)
med_omp = pow2theta(theta_omp)

t1 = st.norm.ppf((1 - beta)/2)
t2 = st.norm.ppf((1 + beta)/2)

asymptotic_med_right = med_omp - t1 * med_omp * np.log(2) / (np.sqrt(n) * (theta_omp - 1))
asymptotic_med_left  = med_omp - t2 * med_omp * np.log(2) / (np.sqrt(n) * (theta_omp - 1))
asymptotic_med_l     = asymptotic_med_right - asymptotic_med_left

print(f"Медиана: {med:.5f}\nОценка медианы: {med_omp:.5f}")
print(f"Асимптотический доверительный интервал для медианы: {asymptotic_med_left:.5f} ≲ theta ≲ {asymptotic_med_right:.5f}")
print(f"Длина интервала l \u2248 {asymptotic_med_l:.5f}")

Медиана: 1.02930
Оценка медианы: 1.02826
Асимптотический доверительный интервал для медианы: 1.02264 ≲ theta ≲ 1.03387
Длина интервала l ≈ 0.01123


In [40]:
# Асимптотический доверительный интервал для theta

asymptotic_left  = theta_omp - t2 / (np.sum(np.log(X)) / np.sqrt(n))
asymptotic_right = theta_omp - t1 / (np.sum(np.log(X)) / np.sqrt(n))
asymptotic_l     = asymptotic_right - asymptotic_left

print(f"Theta: {theta}\nОценка theta: {theta_omp:.5f}")
print(f"Асимптотический доверительный интервал для theta: {asymptotic_left:.5f} ≲ theta ≲ {asymptotic_right:.5f}")
print(f"Длина интервала l \u2248 {asymptotic_l:.5f}")

Theta: 25
Оценка theta: 25.87551
Асимптотический доверительный интервал для theta: 21.00000 ≲ theta ≲ 30.75102
Длина интервала l ≈ 9.75102


**t)** Бутстраповский доверительный интервал

In [41]:
N = 1000

def np_bootstrap(x, N):
    arr = list()
    n = len(x)

    for _ in range(N):
        sample = np.random.choice(x, size=n, replace=True)        
        arr += [(1 + 1 / (np.mean(np.log(sample)))) - theta]

    return sorted(np.array(arr))

bootstrap_array = np_bootstrap(X, N)

t1 = int((1 - beta)/2 * N - 1)
t2 = int((1 + beta)/2 * N - 1)

bootstrap_left = theta_omp - bootstrap_array[t2]
bootstrap_right = theta_omp - bootstrap_array[t1]
bootstrap_l_nonp = bootstrap_right - bootstrap_left

print(f"Непараметрический бутстраповский доверительный интервал для theta (ОМП): {bootstrap_left:.5f} ≲ theta ≲ {bootstrap_right:.5f}")
print(f"Длина интервала l \u2248 {bootstrap_l_nonp:.5f}")

Непараметрический бутстраповский доверительный интервал для theta (ОМП): 18.99819 ≲ theta ≲ 29.73015
Длина интервала l ≈ 10.73196


In [42]:
N = 50000

def p_bootstrap(x, N):
    arr = list()
    n = len(x)

    for _ in range(N):
        sample = pareto(st.uniform(loc=0, scale=1).rvs(size=n), theta=theta_omp)        
        arr += [(1 + 1 / (np.mean(np.log(sample)))) - theta]

    return sorted(np.array(arr))

bootstrap_array = p_bootstrap(X, N)

t1 = int((1 - beta)/2 * N - 1)
t2 = int((1 + beta)/2 * N - 1)

bootstrap_left = theta_omp - bootstrap_array[t2]
bootstrap_right = theta_omp - bootstrap_array[t1]
bootstrap_l_param = bootstrap_right - bootstrap_left

print(f"Параметрический бутстраповский доверительный интервал для theta (ОМП): {bootstrap_left:.5f} ≲ theta ≲ {bootstrap_right:.5f}")
print(f"Длина интервала l \u2248 {bootstrap_l_param:.5f}")

Параметрический бутстраповский доверительный интервал для theta (ОМП): 19.28707 ≲ theta ≲ 29.22074
Длина интервала l ≈ 9.93367


**h)** Сравнить все интервалы

In [43]:
from tabulate import tabulate

lengths = {
    "Асимптотический": asymptotic_l,
    "Параметрический бутстрап": bootstrap_l_param,
    "Непараметрический бутстрап": bootstrap_l_nonp
}

sorted_lengths = sorted([(value, name) for name, value in lengths.items()])

table_data = []
for i, (length, name) in enumerate(sorted_lengths, start=1):
    table_data.append([i, name, np.round(length, 3)])

print(tabulate(table_data, headers=["№", "Метод", "Длина (l)"], tablefmt="fancy_outline"))

╒═════╤════════════════════════════╤═════════════╕
│   № │ Метод                      │   Длина (l) │
╞═════╪════════════════════════════╪═════════════╡
│   1 │ Асимптотический            │       9.751 │
│   2 │ Параметрический бутстрап   │       9.934 │
│   3 │ Непараметрический бутстрап │      10.732 │
╘═════╧════════════════════════════╧═════════════╛
